# Walmart Sales Prediction

**Author:** Gouthami Mandapelli  
**Submission:** AICTE / BharatSkills Internship Project  
**Dataset:** Kaggle — Walmart Store Sales Forecasting  

---

This notebook contains the **complete project code** for the Walmart Weekly Sales Prediction project, covering all stages from raw data loading to final predictions and business insights.

## 1. Problem Statement

Walmart operates 45 stores across multiple formats and regions. Accurate **weekly sales forecasts** at the store/department level are critical for:

- **Inventory planning** — stock the right quantities before demand peaks
- **Staffing decisions** — match labour to expected customer volume
- **Promotion planning** — schedule MarkDown events effectively
- **Store-level resource allocation** — prioritise high-volume locations
- **Seasonal preparation** — prepare well in advance for Thanksgiving and Christmas

**Target variable:** `Weekly_Sales` — the weekly revenue per store/department combination.

**Evaluation metric:** Weighted Mean Absolute Error (WMAE), where **holiday weeks receive 5× weight** to reflect their operational significance.

$$\text{WMAE} = \frac{\sum_{i} w_i \cdot |y_i - \hat{y}_i|}{\sum_i w_i}, \quad w_i = 5 \text{ if holiday, else } 1$$

## 2. Objective

Build a complete, reproducible machine-learning pipeline that:

1. Profiles, cleans, and prepares the four raw datasets
2. Performs exploratory analysis to understand sales patterns
3. Engineers a leakage-free feature set using only historical information
4. Trains and validates models using strict chronological (walk-forward) CV
5. Generates 115,064 final weekly sales predictions for the test period
6. Derives data-driven business insights to support operational decisions

## 3. Import Libraries

In [ ]:
import os
import time
import math
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)
RANDOM_STATE = 42
print('Libraries loaded successfully.')
print('pandas', pd.__version__, '| numpy', np.__version__)

## 4. Load Dataset

Four CSV files are provided by the Kaggle Walmart Store Sales Forecasting competition.

| File | Description |
|------|-------------|
| `train.csv` | Historical weekly sales per Store/Dept (2010-02-05 to 2012-10-26) |
| `test.csv` | Store/Dept/Date combinations to predict — **no target values** |
| `features.csv` | Weekly store-level features: Temperature, Fuel Price, MarkDowns 1–5, CPI, Unemployment |
| `stores.csv` | Store metadata: Type (A/B/C) and Size (sq ft) |

In [ ]:
# Load all four original datasets (never modified)
train    = pd.read_csv('train.csv')
test     = pd.read_csv('test.csv')
features = pd.read_csv('features.csv')
stores   = pd.read_csv('stores.csv')

print('train.csv   :', train.shape,    '| columns:', list(train.columns))
print('test.csv    :', test.shape,     '| columns:', list(test.columns))
print('features.csv:', features.shape, '| columns:', list(features.columns))
print('stores.csv  :', stores.shape,   '| columns:', list(stores.columns))

In [ ]:
# Sample rows from each dataset
print('--- train.csv (first 3 rows) ---')
print(train.head(3).to_string(index=False))
print()
print('--- test.csv (first 3 rows) ---')
print(test.head(3).to_string(index=False))
print()
print('--- features.csv (first 3 rows) ---')
print(features.head(3).to_string(index=False))
print()
print('--- stores.csv ---')
print(stores.to_string(index=False))

## 5. Data Understanding

In [ ]:
# Convert Date columns to datetime
for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

# Date ranges
print('Train date range :', train['Date'].min().date(), '->', train['Date'].max().date())
print('Test  date range :', test['Date'].min().date(),  '->', test['Date'].max().date())
print('Features range   :', features['Date'].min().date(), '->', features['Date'].max().date())
print()
print('Unique stores in train :', train['Store'].nunique())
print('Unique depts  in train :', train['Dept'].nunique())
print('Unique (Store,Dept)    :', train.groupby(['Store','Dept']).ngroups)
print()
print('Store types (from stores.csv):')
print(stores['Type'].value_counts().to_string())

In [ ]:
# Missing value analysis
print('=== Missing Values ===')
for df, name in [(train,'train'),(test,'test'),(features,'features'),(stores,'stores')]:
    mv = df.isnull().sum()
    mv = mv[mv > 0]
    if mv.empty:
        print(f'{name}: No missing values')
    else:
        print(f'{name}:')
        for col, cnt in mv.items():
            print(f'  {col}: {cnt:,} missing ({cnt/len(df)*100:.1f}%)')

In [ ]:
# Duplicate checks
print('Duplicate (Store+Dept+Date) in train :', train.duplicated(subset=['Store','Dept','Date']).sum())
print('Duplicate (Store+Dept+Date) in test  :', test.duplicated(subset=['Store','Dept','Date']).sum())
print('Duplicate (Store+Date) in features   :', features.duplicated(subset=['Store','Date']).sum())
print()
# Target distribution
print('=== Weekly_Sales Statistics ===')
ws = train['Weekly_Sales']
print(ws.describe().round(2).to_string())
print(f'Skewness : {ws.skew():.4f}')
print(f'Negative rows : {(ws < 0).sum():,} (product returns)')
print(f'Holiday weeks (IsHoliday=True) : {train["IsHoliday"].sum():,}')

## 6. Data Cleaning

The cleaning pipeline:
1. Merge `train`/`test` with `features` on `(Store, Date)` — left join
2. Join with `stores` on `Store` to add `Type` and `Size`
3. Resolve duplicate `IsHoliday` columns (values agree 100%)
4. Fill `MarkDown1–5` NaN → `0` (structural zeros — no promotion before Nov 2011)
5. Fill `CPI`/`Unemployment` NaN via per-store forward-fill (leakage-safe)
6. Retain negative `Weekly_Sales` (legitimate product returns)

> Original CSV files (`train.csv`, `test.csv`, `features.csv`, `stores.csv`) are **never modified**.

In [ ]:
# ── Step 1: Convert dates ──────────────────────────────────────────────────
for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

# ── Step 2: Merge train + features + stores ────────────────────────────────
train_c = (train
           .merge(features, on=['Store','Date'], how='left',
                  suffixes=('_sales','_feat'))
           .merge(stores, on='Store', how='left'))

test_c  = (test
           .merge(features, on=['Store','Date'], how='left',
                  suffixes=('_sales','_feat'))
           .merge(stores, on='Store', how='left'))

print('train after merge:', train_c.shape, '| rows gained:', len(train_c)-len(train))
print('test  after merge:', test_c.shape,  '| rows gained:', len(test_c)-len(test))

In [ ]:
# ── Step 3: Resolve duplicate IsHoliday columns ───────────────────────────
for df, label in [(train_c,'train'),(test_c,'test')]:
    if 'IsHoliday_sales' in df.columns and 'IsHoliday_feat' in df.columns:
        mismatch = (df['IsHoliday_sales'] != df['IsHoliday_feat']).sum()
        print(f'{label}: IsHoliday mismatch rows = {mismatch} (expected 0)')
        df.rename(columns={'IsHoliday_sales':'IsHoliday'}, inplace=True)
        df.drop(columns=['IsHoliday_feat'], inplace=True)

# ── Step 4: Fill MarkDown NaN with 0 ─────────────────────────────────────
md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
for df in [train_c, test_c]:
    for c in md_cols:
        df[c] = df[c].fillna(0)
print('MarkDown NaN filled with 0 (no promotion active = structural zero)')

# ── Step 5: CPI & Unemployment — per-store forward-fill ──────────────────
for df, label in [(train_c,'train'),(test_c,'test')]:
    df.sort_values(['Store','Date'], inplace=True)
    for c in ['CPI','Unemployment']:
        before = df[c].isna().sum()
        df[c] = df.groupby('Store')[c].transform(lambda s: s.ffill())
        after  = df[c].isna().sum()
        if before > 0:
            print(f'{label} {c}: {before} NaN -> {after} after ffill')

# ── Step 6: Sort chronologically ──────────────────────────────────────────
train_c = train_c.sort_values(['Store','Dept','Date']).reset_index(drop=True)
test_c  = test_c.sort_values(['Store','Dept','Date']).reset_index(drop=True)

print('\nFinal cleaned shapes:')
print('  train_cleaned:', train_c.shape)
print('  test_cleaned :', test_c.shape)
print('  Remaining NaN in train:', train_c.isnull().sum().sum())
print('  Remaining NaN in test :', test_c.isnull().sum().sum())
print(f'  Negative Weekly_Sales kept: {(train_c["Weekly_Sales"]<0).sum():,} rows')

In [ ]:
# Save cleaned datasets
train_c.to_csv('train_cleaned.csv', index=False)
test_c.to_csv('test_cleaned.csv',   index=False)
print('Saved: train_cleaned.csv', train_c.shape)
print('Saved: test_cleaned.csv ', test_c.shape)

## 7. Data Preparation

- Extract calendar features: `Year`, `Month`, `Week` (ISO), `Quarter`
- `DayOfWeek` is created, verified (constant = 4 = Friday for every row), then dropped
- Add `IsHoliday_int` (boolean → 0/1 integer)
- Add `Is_Known_Holiday` for the four annual Walmart holiday weeks: Super Bowl (Week 6), Labour Day (Week 36), Thanksgiving (Week 47), Christmas (Week 52)

**No random splitting.** Chronological validation splits are defined here.

In [ ]:
train_p = pd.read_csv('train_cleaned.csv', parse_dates=['Date'])
test_p  = pd.read_csv('test_cleaned.csv',  parse_dates=['Date'])

# Calendar features
for df in [train_p, test_p]:
    df['Year']      = df['Date'].dt.year
    df['Month']     = df['Date'].dt.month
    df['Week']      = df['Date'].dt.isocalendar().week.astype(int)
    df['Quarter']   = df['Date'].dt.quarter
    df['DayOfWeek'] = df['Date'].dt.dayofweek

# Verify and drop DayOfWeek (constant = 4 for all rows)
dow_train_unique = train_p['DayOfWeek'].unique()
dow_test_unique  = test_p['DayOfWeek'].unique()
print('DayOfWeek unique values — train:', dow_train_unique, '| test:', dow_test_unique)
print('All Fridays (4)?', (dow_train_unique == [4]).all() and (dow_test_unique == [4]).all())
train_p.drop(columns=['DayOfWeek'], inplace=True)
test_p.drop(columns=['DayOfWeek'], inplace=True)
print('DayOfWeek dropped (zero variance, adds no predictive information).')

In [ ]:
# Save prepared datasets
train_p.to_csv('train_prepared.csv', index=False)
test_p.to_csv('test_prepared.csv',   index=False)
print('train_prepared.csv:', train_p.shape)
print('test_prepared.csv :', test_p.shape)

# Chronological validation strategy
print('\n=== Chronological Validation Strategy ===')
print('Primary split:')
print('  Train  : 2010-02-05 to 2012-07-27')
print('  Val    : 2012-08-03 to 2012-10-26')
print()
print('Walk-forward 3-fold CV (expanding window):')
FOLDS = [
    ('Fold 1', '2011-10-21', '2011-10-28', '2012-02-03'),
    ('Fold 2', '2012-01-27', '2012-02-03', '2012-05-04'),
    ('Fold 3', '2012-04-27', '2012-05-04', '2012-10-26'),
]
for name, te, vs, ve in FOLDS:
    tr_n  = (train_p['Date'] <= te).sum()
    val_n = ((train_p['Date'] >= vs) & (train_p['Date'] <= ve)).sum()
    print(f'  {name}: train<={te} ({tr_n:,} rows) | val {vs}->{ve} ({val_n:,} rows)')

## 8. Exploratory Data Analysis

EDA is performed on `train_cleaned.csv`. Pre-generated plots from `eda_plots/` are displayed alongside the analysis code.

In [ ]:
train_eda = pd.read_csv('train_cleaned.csv', parse_dates=['Date'])
ws = train_eda['Weekly_Sales']

print('=== Target Variable: Weekly_Sales ===')
print(ws.describe().round(2).to_string())
print(f'Skewness : {ws.skew():.4f}  (heavily right-skewed)')
print(f'Mean > Median: {ws.mean():.0f} > {ws.median():.0f} -- driven by high-volume depts')
print(f'Negative rows : {(ws < 0).sum():,} (product returns, retained)')

In [ ]:
# Distribution and trend plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fpath, title in zip(
    axes,
    ['eda_plots/01_target_distribution.png', 'eda_plots/03_weekly_total_sales_trend.png'],
    ['Weekly_Sales Distribution', 'Weekly Total Sales Trend']
):
    img = mpimg.imread(fpath)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Seasonality analysis
train_eda['Year']  = train_eda['Date'].dt.year
train_eda['Month'] = train_eda['Date'].dt.month
train_eda['Week']  = train_eda['Date'].dt.isocalendar().week.astype(int)

print('=== Monthly Average Sales ===')
monthly = train_eda.groupby('Month')['Weekly_Sales'].mean().round(2)
print(monthly.to_string())
print()
print('Peak months: Nov (11) and Dec (12) -- Thanksgiving & Christmas')
print('Trough: Jan–Feb after holiday season')

In [ ]:
# Seasonality plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fpath, title in zip(
    axes,
    ['eda_plots/05_monthly_seasonality.png', 'eda_plots/06_week_of_year_seasonality.png'],
    ['Monthly Seasonality', 'Week-of-Year Seasonality']
):
    img = mpimg.imread(fpath)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Store and Department analysis
store_avg  = train_eda.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)
dept_avg   = train_eda.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False)

print('Top 5 Stores by Average Weekly Sales:')
print(store_avg.head(5).round(2).to_string())
print()
print('Bottom 5 Stores by Average Weekly Sales:')
print(store_avg.tail(5).round(2).to_string())
print()
print('Top 10 Departments by Average Weekly Sales:')
print(dept_avg.head(10).round(2).to_string())

In [ ]:
# Store and Dept plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fpath, title in zip(
    axes,
    ['eda_plots/07_store_mean_sales.png', 'eda_plots/09_dept_top20_mean_sales.png'],
    ['Store Mean Sales', 'Top 20 Departments Mean Sales']
):
    img = mpimg.imread(fpath)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Store Type analysis
type_stats = train_eda.groupby('Type')['Weekly_Sales'].agg(
    Count='count', Mean='mean', Median='median', Std='std'
).round(2)
print('Store Type Comparison:')
print(type_stats.to_string())
print()
print('Type A (largest) has highest avg sales.')
print('Type C (smallest) has lowest avg sales.')
print('These differences are consistent with store size, not causal claims.')

In [ ]:
# Holiday analysis
hol_stats = train_eda.groupby('IsHoliday')['Weekly_Sales'].agg(
    Count='count', Mean='mean', Median='median'
).round(2)
print('Holiday vs Non-Holiday Sales:')
print(hol_stats.to_string())
hol_lift = (train_eda[train_eda['IsHoliday']==True]['Weekly_Sales'].mean() /
            train_eda[train_eda['IsHoliday']==False]['Weekly_Sales'].mean() - 1) * 100
print(f'Holiday mean lift: +{hol_lift:.1f}%')
print('Holiday weeks receive 5x weight in WMAE evaluation.')

In [ ]:
# Holiday and external feature plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, fpath, title in zip(
    axes,
    ['eda_plots/13_holiday_boxplot.png',
     'eda_plots/15_external_feat_distributions.png',
     'eda_plots/19_correlation_heatmap.png'],
    ['Holiday Boxplot', 'External Features', 'Correlation Heatmap']
):
    img = mpimg.imread(fpath)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# MarkDown analysis
md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
print('MarkDown active (>0) percentage in training:')
for c in md_cols:
    pct = (train_eda[c] > 0).mean() * 100
    print(f'  {c}: {pct:.1f}% of weeks active')
print()
print('MarkDown1 <-> MarkDown4 correlation:',
      round(train_eda['MarkDown1'].corr(train_eda['MarkDown4']), 3))
print('Note: MarkDowns were absent before Nov 2011 (structural zeros).')

In [ ]:
# Size vs Sales scatter insight
size_corr = train_eda.merge(stores[['Store','Size']], on='Store', how='left') \
              .groupby('Store')['Weekly_Sales'].mean() \
              .reset_index()
size_corr = size_corr.merge(stores[['Store','Size']], on='Store')
r = size_corr['Weekly_Sales'].corr(size_corr['Size'])
print(f'Pearson correlation — Store Size vs Mean Weekly Sales: {r:.4f}')
print('Strong positive correlation: larger stores tend to generate higher weekly revenue.')

# Display size vs sales plot
img = mpimg.imread('eda_plots/18_sales_vs_size.png')
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img); ax.axis('off'); ax.set_title('Store Size vs Mean Weekly Sales')
plt.tight_layout(); plt.show()

## 9. Feature Engineering

### Approved Stage-1 Feature Set (25 features)

| Category | Features |
|----------|----------|
| Identifiers | `Store`, `Dept` |
| Calendar | `Year`, `Month`, `Week`, `Quarter` |
| Store info | `Type` (label-encoded A→0, B→1, C→2), `Size` |
| Holiday | `IsHoliday_int`, `Is_Known_Holiday` |
| Economic | `Temperature`, `Fuel_Price`, `CPI`, `Unemployment` |
| Promotions | `MarkDown1`–`5`, `MarkDown1_active`–`MarkDown5_active` |
| Lag feature | `lag_52` (same store/dept, 52 weeks prior) |

**Excluded from Stage-1:** `lag_1`, `lag_2`, `lag_4`, `rolling_mean_*`, `rolling_std_*`  
**Leakage-safe:** `lag_52` always looks 52 weeks back into the training period. Test starts Nov 2012; 52w prior = Nov 2011 = within training.

In [ ]:
train_f = pd.read_csv('train_prepared.csv', parse_dates=['Date'])
test_f  = pd.read_csv('test_prepared.csv',  parse_dates=['Date'])

train_f = train_f.sort_values(['Store','Dept','Date']).reset_index(drop=True)
test_f  = test_f.sort_values(['Store','Dept','Date']).reset_index(drop=True)

# ── MarkDown activity flags ───────────────────────────────────────────────
md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
for df in [train_f, test_f]:
    for i, c in enumerate(md_cols, 1):
        df[f'MarkDown{i}_active'] = (df[c] != 0).astype(int)
print('MarkDown activity flags created.')

# ── Holiday features ──────────────────────────────────────────────────────
KNOWN_HOLIDAY_WEEKS = {6, 36, 47, 52}  # Super Bowl, Labour Day, Thanksgiving, Christmas
for df in [train_f, test_f]:
    df['IsHoliday_int']    = df['IsHoliday'].astype(int)
    df['Is_Known_Holiday'] = df['Week'].isin(KNOWN_HOLIDAY_WEEKS).astype(int)
print('Holiday integer and known-holiday flags created.')

In [ ]:
# ── lag_52 for training set (shift-based, within Store+Dept group) ────────
# Rule: row at date T uses Weekly_Sales from T-52 weeks — no current row used.
train_f['lag_52'] = (train_f
                     .groupby(['Store','Dept'])['Weekly_Sales']
                     .shift(52))

print(f'lag_52 NaN in train: {train_f["lag_52"].isna().sum():,} '
      f'(first 52 rows per group, expected)')

# Leakage verification: lag_52 at row i must equal Weekly_Sales at row i-52
leak_count = 0
for (s, d), grp in train_f.groupby(['Store','Dept']):
    grp = grp.sort_values('Date').reset_index(drop=True)
    expected = grp['Weekly_Sales'].shift(52)
    mismatch = (~np.isclose(grp['lag_52'].fillna(-999), expected.fillna(-999))).sum()
    leak_count += mismatch
print(f'Leakage check — mismatches: {leak_count} (expected 0)')

In [ ]:
# ── lag_52 for test set (date-indexed lookup into training) ──────────────
# For any test date T, T-52w falls in the training period.
ONE_WEEK = np.timedelta64(7, 'D')
train_lookup = {}
for (s, d), grp in train_f.groupby(['Store','Dept']):
    g = grp.sort_values('Date')
    train_lookup[(s, d)] = {'dates': g['Date'].values,
                             'sales': g['Weekly_Sales'].values}

test_f['lag_52'] = np.nan
for (s, d), idx in test_f.groupby(['Store','Dept']).groups.items():
    if (s, d) not in train_lookup:
        continue
    grp    = test_f.loc[idx].sort_values('Date')
    dates  = train_lookup[(s,d)]['dates']
    sales  = train_lookup[(s,d)]['sales']
    vals   = []
    for tdate in grp['Date'].values:
        tgt = tdate - 52 * ONE_WEEK
        i   = np.searchsorted(dates, tgt, side='left')
        vals.append(sales[i] if i < len(dates) and dates[i] == tgt else np.nan)
    test_f.loc[idx, 'lag_52'] = vals

non_null = test_f['lag_52'].notna().sum()
print(f'lag_52 populated in test: {non_null:,} / {len(test_f):,} '
      f'({non_null/len(test_f)*100:.1f}%)')
print('NaN rows are test-only (Store,Dept) pairs with no training history — imputed later.')

In [ ]:
# ── Save feature-engineered datasets ──────────────────────────────────────
FEATURES = [
    'Store','Dept','Year','Month','Week','Quarter','Type',
    'IsHoliday_int','Is_Known_Holiday','Temperature','Fuel_Price',
    'MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5',
    'MarkDown1_active','MarkDown2_active','MarkDown3_active',
    'MarkDown4_active','MarkDown5_active','CPI','Unemployment','Size','lag_52'
]
TARGET = 'Weekly_Sales'

# Stage-2 lag/rolling columns (in train for completeness, NaN in test by design)
for n in [1, 2, 4]:
    train_f[f'lag_{n}'] = train_f.groupby(['Store','Dept'])['Weekly_Sales'].shift(n)
for test_col in ['lag_1','lag_2','lag_4',
                  'rolling_mean_4','rolling_mean_12','rolling_mean_52','rolling_std_4']:
    test_f[test_col] = np.nan
for col_name, window in [('rolling_mean_4',4),('rolling_mean_12',12),
                          ('rolling_mean_52',52),('rolling_std_4',4)]:
    shifted = train_f.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
    if 'std' in col_name:
        train_f[col_name] = shifted.groupby([train_f['Store'],train_f['Dept']]) \
                            .transform(lambda s: s.rolling(window, min_periods=1).std())
    else:
        train_f[col_name] = shifted.groupby([train_f['Store'],train_f['Dept']]) \
                            .transform(lambda s: s.rolling(window, min_periods=1).mean())

train_f.to_csv('train_features.csv', index=False)
test_f.to_csv('test_features.csv',   index=False)
print('Saved: train_features.csv', train_f.shape)
print('Saved: test_features.csv ', test_f.shape)
print('Stage-1 features:', len(FEATURES))

## 10. Train / Validation Strategy

**No random splitting is used anywhere in this project.**

All splits are strictly chronological to prevent data leakage and correctly simulate a real-world forecasting scenario.

### WMAE Formula
$$\text{WMAE} = \frac{\sum_i w_i \cdot |y_i - \hat{y}_i|}{\sum_i w_i}$$
where $w_i = 5$ if `IsHoliday = True`, else $w_i = 1$.

In [ ]:
# ── WMAE and metric helper functions ─────────────────────────────────────
def wmae(y_true, y_pred, is_holiday):
    '''Weighted MAE: holiday weeks weight 5, non-holiday weight 1.'''
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def encode_type(series):
    '''Label-encode Type: A->0, B->1, C->2.'''
    return series.map({'A':0,'B':1,'C':2}).astype(int)

def impute_lag52(train_df, target_df):
    '''Leakage-safe imputation: (Store,Dept) median > Dept median > global median.
    Statistics computed from train_df ONLY.'''
    pair_med   = train_df.dropna(subset=['lag_52']).groupby(['Store','Dept'])['lag_52'].median()
    dept_med   = train_df.dropna(subset=['lag_52']).groupby('Dept')['lag_52'].median()
    global_med = float(train_df['lag_52'].median())
    series   = target_df['lag_52'].copy()
    nan_mask = series.isna()
    if not nan_mask.any():
        return series
    for idx in target_df.index[nan_mask]:
        s, d = target_df.at[idx,'Store'], target_df.at[idx,'Dept']
        if (s,d) in pair_med.index:  series.at[idx] = pair_med[(s,d)]
        elif d in dept_med.index:    series.at[idx] = dept_med[d]
        else:                        series.at[idx] = global_med
    return series

def get_X_y(df, features, target=None):
    df = df.copy()
    df['Type'] = encode_type(df['Type'])
    X = df[features]
    y = df[target] if target else None
    return X, y

print('Helper functions defined: wmae, rmse_score, encode_type, impute_lag52, get_X_y')

## 11. Model Training

Three models are trained and compared using walk-forward cross-validation.

| Model | Description |
|-------|-------------|
| **Lag-52 Baseline** | Naive: predict = same store/dept last-year same week |
| **Random Forest** | 150 trees, max_depth=20, min_samples_leaf=4 |
| **HistGradientBoosting** | 500 iters, lr=0.05, early stopping |

> **Saved validation results are loaded** to avoid re-running the expensive ~700-second training. The complete training code is shown below for full transparency.

In [ ]:
# Load train_features.csv (used for training)
train_feat = pd.read_csv('train_features.csv', parse_dates=['Date'])
print('train_features shape:', train_feat.shape)
print('date range:', train_feat['Date'].min().date(), '->', train_feat['Date'].max().date())

In [ ]:
# ── Walk-Forward Fold Configuration ────────────────────────────────────────
FOLD_DEFS = [
    dict(name='Fold 1', train_end='2011-10-21', val_start='2011-10-28', val_end='2012-02-03'),
    dict(name='Fold 2', train_end='2012-01-27', val_start='2012-02-03', val_end='2012-05-04'),
    dict(name='Fold 3', train_end='2012-04-27', val_start='2012-05-04', val_end='2012-10-26'),
]
PRIMARY_TRAIN_END = '2012-07-27'
PRIMARY_VAL_START = '2012-08-03'
PRIMARY_VAL_END   = '2012-10-26'

print('Fold sizes:')
for fold in FOLD_DEFS:
    tr_n  = (train_feat['Date'] <= fold['train_end']).sum()
    val_n = ((train_feat['Date'] >= fold['val_start']) &
             (train_feat['Date'] <= fold['val_end'])).sum()
    print(f"  {fold['name']}: train={tr_n:,} | val={val_n:,}")

In [ ]:
# ── Complete model training code (run once; results saved to disk) ─────────
# NOTE: This code runs the full training. On first run it will take ~10 minutes.
# After completion, results are saved and loaded in the next cell.
# To skip retraining, set RUN_TRAINING = False.

RUN_TRAINING = False  # Set True to retrain; False loads saved results

if RUN_TRAINING:
    import os
    os.makedirs('model_outputs', exist_ok=True)
    cv_records = []

    for fold in FOLD_DEFS:
        fname    = fold['name']
        fold_tr  = train_feat[train_feat['Date'] <= fold['train_end']].copy().reset_index(drop=True)
        fold_val = train_feat[(train_feat['Date'] >= fold['val_start']) &
                              (train_feat['Date'] <= fold['val_end'])].copy().reset_index(drop=True)

        fold_tr['lag_52']  = impute_lag52(fold_tr, fold_tr)
        fold_val['lag_52'] = impute_lag52(fold_tr, fold_val)

        X_tr,  y_tr  = get_X_y(fold_tr,  FEATURES, TARGET)
        X_val, y_val = get_X_y(fold_val, FEATURES, TARGET)
        is_hol_val   = fold_val['IsHoliday'].values.astype(bool)

        # Lag-52 Baseline
        bl_p = fold_val['lag_52'].values
        cv_records.append({'Fold':fname,'Model':'Lag-52 Baseline',
                           'WMAE':round(wmae(y_val.values, bl_p, is_hol_val),4),
                           'MAE':round(mean_absolute_error(y_val.values, bl_p),4),
                           'RMSE':round(rmse_score(y_val.values, bl_p),4)})
        print(f'{fname} | Lag-52 Baseline WMAE={cv_records[-1]["WMAE"]:,.2f}')

        # Random Forest
        rf = RandomForestRegressor(n_estimators=150, max_depth=20,
                                    min_samples_leaf=4, max_features=0.7,
                                    n_jobs=-1, random_state=RANDOM_STATE)
        t0 = time.time()
        rf.fit(X_tr, y_tr)
        rf_p = rf.predict(X_val)
        cv_records.append({'Fold':fname,'Model':'Random Forest',
                           'WMAE':round(wmae(y_val.values,rf_p,is_hol_val),4),
                           'MAE':round(mean_absolute_error(y_val.values,rf_p),4),
                           'RMSE':round(rmse_score(y_val.values,rf_p),4),
                           'Train_s':round(time.time()-t0,1)})
        print(f'{fname} | Random Forest    WMAE={cv_records[-1]["WMAE"]:,.2f} ({cv_records[-1].get("Train_s",0):.1f}s)')

        # HistGradientBoosting
        hgb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05,
            max_leaf_nodes=63, min_samples_leaf=20, l2_regularization=0.1,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=20, random_state=RANDOM_STATE)
        t0 = time.time()
        hgb.fit(X_tr, y_tr)
        hgb_p = hgb.predict(X_val)
        cv_records.append({'Fold':fname,'Model':'HistGradBoost',
                           'WMAE':round(wmae(y_val.values,hgb_p,is_hol_val),4),
                           'MAE':round(mean_absolute_error(y_val.values,hgb_p),4),
                           'RMSE':round(rmse_score(y_val.values,hgb_p),4),
                           'Train_s':round(time.time()-t0,1)})
        print(f'{fname} | HistGradBoost    WMAE={cv_records[-1]["WMAE"]:,.2f} ({cv_records[-1].get("Train_s",0):.1f}s)')

    pd.DataFrame(cv_records).to_csv('model_outputs/_nb_cv_records.csv', index=False)
    print('Training complete. Results saved.')
else:
    print('RUN_TRAINING=False — loading saved validation results from disk.')

## 12. Model Evaluation

Saved validation results from the completed Stage 6 run are loaded. WMAE is the primary model-selection metric.

In [ ]:
# Load saved validation results (produced by model_training.py)
val_results = pd.read_csv('model_outputs/model_validation_results.csv')
print('=== Stage 6 Validation Results ===')
display_cols = ['Model','Fold 1','Fold 2','Fold 3',
                'Avg_CV_WMAE','Primary_WMAE','Primary_MAE','Primary_RMSE']
print(val_results[display_cols].to_string(index=False))

In [ ]:
# Visualise average CV WMAE comparison
fig, ax = plt.subplots(figsize=(9, 4))
models  = val_results['Model'].str.strip().tolist()
avg_w   = val_results['Avg_CV_WMAE'].tolist()
colors  = ['#3b82d4' if m=='Lag-52 Baseline' else '#94a3b8' for m in models]
bars    = ax.barh(models, avg_w, color=colors, edgecolor='white')
for bar, val in zip(bars, avg_w):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=10)
ax.set_xlabel('Average CV WMAE (lower is better)', fontsize=11)
ax.set_title('Model Comparison — Walk-Forward CV WMAE', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

best_row = val_results.sort_values('Avg_CV_WMAE').iloc[0]
print(f'Selected model : {best_row["Model"].strip()}')
print(f'Avg CV WMAE    : {best_row["Avg_CV_WMAE"]:,.4f}')
print(f'Primary WMAE   : {best_row["Primary_WMAE"]:,.4f}')
print(f'Primary MAE    : {best_row["Primary_MAE"]:,.4f}')
print(f'Primary RMSE   : {best_row["Primary_RMSE"]:,.4f}')

### Selection Rationale

The **Lag-52 Baseline** is selected because it achieves the lowest **average cross-validated WMAE** (2,278.13) across all three chronological folds.

The Random Forest outperformed on the single primary hold-out split (WMAE 1,561) but had a very high Fold-1 WMAE (3,970) due to the **cold-start effect** — Fold-1 training ends before any MarkDown history existed (Nov 2011), severely degrading the RF's performance. The Baseline is immune to this because it uses no learned feature distributions. The three-fold average is a more reliable model-selection criterion than a single hold-out window.

## 13. Final Model

The Lag-52 Baseline is retrained on the **complete training dataset** (421,570 rows) and saved as `model_outputs/final_model.pkl`.

The Lag-52 Baseline prediction is: $\hat{y}_t = \text{Weekly\_Sales}_{t-52\text{w}}$

No sklearn fitting is required. The model is saved as a descriptor dict with full metadata.

In [ ]:
# Load full training data and impute lag_52 using full training set
full_train = pd.read_csv('train_features.csv', parse_dates=['Date'])
full_train = full_train.sort_values(['Store','Dept','Date']).reset_index(drop=True)

full_train['lag_52'] = impute_lag52(full_train, full_train)

print(f'Full training rows : {len(full_train):,}')
print(f'lag_52 NaN after imputation : {full_train["lag_52"].isna().sum()}')
print(f'Training date range: {full_train["Date"].min().date()} to {full_train["Date"].max().date()}')

In [ ]:
# Save final model
import os
os.makedirs('model_outputs', exist_ok=True)

final_model = {
    'model':    'Lag-52 Baseline',
    'strategy': 'prediction = lag_52 value (same store/dept, 52 weeks prior)',
    'trained_on_rows': len(full_train),
}
final_meta = {
    'selected_model_name': 'Lag-52 Baseline',
    'features': FEATURES,
    'target': TARGET,
    'type_encoding': {'A':0,'B':1,'C':2},
    'training_rows': len(full_train),
    'training_date_range': f'{full_train["Date"].min().date()} to {full_train["Date"].max().date()}',
    'lag52_imputation': 'Store+Dept median > Dept median > global median (training data only)',
}

with open('model_outputs/final_model.pkl','wb') as f:
    pickle.dump(final_model, f)
with open('model_outputs/final_model_meta.pkl','wb') as f:
    pickle.dump(final_meta, f)
# Also update selected_model.pkl so app.py uses the final version
with open('model_outputs/selected_model.pkl','wb') as f:
    pickle.dump(final_model, f)

print('Saved: model_outputs/final_model.pkl')
print('Saved: model_outputs/final_model_meta.pkl')
print('Saved: model_outputs/selected_model.pkl')

# Verify load
with open('model_outputs/final_model.pkl','rb') as f:
    loaded = pickle.load(f)
print('Loaded model:', loaded)

## 14. Final Test Predictions

Predictions are generated for all **115,064 test rows** covering 2012-11-02 to 2013-07-26.

**Prediction = lag_52** (same store/department, same week, one year prior)

> Test-set actual `Weekly_Sales` are unavailable (Kaggle competition format). All predictions are model estimates, not confirmed outcomes.

In [ ]:
# Load test features and impute lag_52
test_feat = pd.read_csv('test_features.csv', parse_dates=['Date'])
test_feat = test_feat.sort_values(['Store','Dept','Date']).reset_index(drop=True)

# Impute lag_52 using FULL training set (no test data used)
test_feat['lag_52'] = impute_lag52(full_train, test_feat)

print(f'Test rows          : {len(test_feat):,}')
print(f'lag_52 NaN in test : {test_feat["lag_52"].isna().sum()} (expected 0)')

In [ ]:
# Pre-prediction verification
assert len(test_feat) == 115064, f'Expected 115064 rows, got {len(test_feat)}'
key_dups = test_feat.duplicated(subset=['Store','Dept','Date']).sum()
assert key_dups == 0, f'Duplicate keys found: {key_dups}'
missing_feats = [f for f in FEATURES if f not in test_feat.columns]
assert len(missing_feats) == 0, f'Missing features: {missing_feats}'
assert test_feat['lag_52'].isna().sum() == 0, 'lag_52 still has NaN'
print('All pre-prediction checks passed.')
print('  row count = 115,064')
print('  no duplicate Store+Dept+Date keys')
print('  all 25 features present')
print('  lag_52 fully imputed')

In [ ]:
# Generate predictions (Lag-52 Baseline: prediction = lag_52)
test_preds = test_feat['lag_52'].values.copy()

# Verify predictions
assert not np.any(np.isnan(test_preds)),  'NaN values in predictions'
assert not np.any(np.isinf(test_preds)),  'Infinite values in predictions'
assert len(test_preds) == 115064
print('Prediction verification passed: no NaN, no Inf, 115,064 rows.')
print(f'  Min prediction : {test_preds.min():,.2f}')
print(f'  Max prediction : {test_preds.max():,.2f}')
print(f'  Mean prediction: {test_preds.mean():,.2f}')

In [ ]:
# Build and save prediction file
pred_df = pd.DataFrame({
    'Store':        test_feat['Store'].values,
    'Dept':         test_feat['Dept'].values,
    'Date':         test_feat['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': np.round(test_preds, 4),
})

pred_df.to_csv('model_outputs/walmart_sales_predictions.csv', index=False)
print('Saved: model_outputs/walmart_sales_predictions.csv')
print('Shape:', pred_df.shape)
print('Columns:', list(pred_df.columns))
print()
print('Preview (first 10 rows):')
print(pred_df.head(10).to_string(index=False))

In [ ]:
# Verify Store+Dept+Date match original test.csv exactly
orig_test = pd.read_csv('test.csv', parse_dates=['Date'])
orig_test['Date_str'] = orig_test['Date'].dt.strftime('%Y-%m-%d')
pred_key = set(zip(pred_df['Store'].astype(int),
                    pred_df['Dept'].astype(int),
                    pred_df['Date']))
test_key = set(zip(orig_test['Store'].astype(int),
                    orig_test['Dept'].astype(int),
                    orig_test['Date_str']))
print(f'Prediction keys : {len(pred_key):,}')
print(f'Test.csv keys   : {len(test_key):,}')
print(f'Match           : {pred_key == test_key}')

## 15. Business Insights

In [ ]:
# ── Overall Sales Pattern ─────────────────────────────────────────────────
ws = full_train['Weekly_Sales']
print('=== Overall Sales Pattern ===')
print(f'Total training records : {len(ws):,}')
print(f'Weekly Sales range     : {ws.min():,.2f} to {ws.max():,.2f}')
print(f'Mean Weekly Sales      : {ws.mean():,.2f}')
print(f'Median Weekly Sales    : {ws.median():,.2f}')
print(f'Skewness               : {ws.skew():.4f} (right-skewed)')
print(f'Negative rows (returns): {(ws<0).sum():,}')

In [ ]:
# ── Store-level insights ─────────────────────────────────────────────────
store_hist = full_train.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)
print('Top 5 Stores by Average Historical Weekly Sales:')
print(store_hist.head(5).round(2).to_string())
print()
print('Bottom 5 Stores by Average Historical Weekly Sales:')
print(store_hist.tail(5).round(2).to_string())
print()
store_sum = pd.read_csv('model_outputs/store_prediction_summary.csv')
print('Top 5 Stores by Predicted Total Sales (test period):')
print(store_sum.head(5).to_string(index=False))

In [ ]:
# ── Department-level insights ─────────────────────────────────────────────
dept_hist = full_train.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False)
print('Top 10 Departments by Average Historical Weekly Sales:')
print(dept_hist.head(10).round(2).to_string())
print()
dept_sum = pd.read_csv('model_outputs/department_prediction_summary.csv')
print('Top 5 Departments by Predicted Total Sales (test period):')
print(dept_sum.head(5).to_string(index=False))

In [ ]:
# ── Store type, holiday, and prediction summary ───────────────────────────
type_stats = full_train.groupby('Type')['Weekly_Sales'].agg(
    Count='count', Mean='mean', Median='median').round(2)
print('=== Store Type Summary ===')
print(type_stats.to_string())
print()
hol_stats = full_train.groupby('IsHoliday')['Weekly_Sales'].agg(
    Count='count', Mean='mean', Median='median').round(2)
print('=== Holiday vs Non-Holiday ===')
print(hol_stats.to_string())
print('(Holiday weeks carry 5x weight in WMAE evaluation.)')
print()
print('=== Final Prediction Summary ===')
print(f'Test rows predicted        : {len(pred_df):,}')
print(f'Total predicted sales      : {pred_df["Weekly_Sales"].sum():,.2f}')
print(f'Average predicted weekly   : {pred_df["Weekly_Sales"].mean():,.2f}')
print(f'Test period                : {pred_df["Date"].min()} to {pred_df["Date"].max()}')

In [ ]:
# ── Business takeaways visualisation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Store type mean sales
type_means = full_train.groupby('Type')['Weekly_Sales'].mean()
axes[0].bar(type_means.index, type_means.values,
            color=['#3b82d4','#7c5cd8','#94a3b8'], edgecolor='white')
axes[0].set_title('Average Weekly Sales by Store Type', fontweight='bold')
axes[0].set_xlabel('Store Type'); axes[0].set_ylabel('Mean Weekly Sales ($)')
for i, (t, v) in enumerate(type_means.items()):
    axes[0].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9)

# Monthly seasonality
monthly = full_train.groupby('Month')['Weekly_Sales'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar(range(1,13), [monthly.get(m,0) for m in range(1,13)],
            color='#3b82d4', edgecolor='white')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_names)
axes[1].set_title('Monthly Average Weekly Sales (Seasonality)', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Mean Weekly Sales ($)')

plt.tight_layout(); plt.show()
print('Nov-Dec peak reflects Thanksgiving + Christmas holiday surge.')

## 16. Conclusion

### What Was Built
A complete, reproducible machine-learning pipeline for Walmart weekly store/department sales forecasting, progressing through nine structured stages.

### Dataset
Four CSV files (421,570 training rows, 115,064 test rows, 45 stores, 81 departments) from the Kaggle Walmart Store Sales Forecasting competition.

### Preprocessing
- Left-join merge of train/test with features and stores
- MarkDown NaN filled with 0 (structural — no promotion active)
- CPI/Unemployment forward-filled per store (leakage-safe)
- Negative Weekly_Sales retained (legitimate returns)

### Feature Engineering
25 Stage-1 features including calendar, store metadata, holiday flags, MarkDown activity indicators, economic/environmental variables, and **lag_52** (same store/dept, same week, one year prior). lag_52 is the only lag feature used — fully leakage-safe.

### Model & Validation
- **Selected model:** Lag-52 Baseline
- **Validation:** Walk-forward 3-fold chronological CV (no random splitting)
- **Average CV WMAE:** 2,278.13
- **Primary hold-out WMAE:** 1,893.62 | **MAE:** 1,867.98 | **RMSE:** 3,936.34

### Final Predictions
115,064 predictions for 2012-11-02 to 2013-07-26 saved to `model_outputs/walmart_sales_predictions.csv`.

### Business Use Cases
Inventory pre-positioning, staffing for holiday peak weeks, MarkDown timing, store-level resource allocation, and seasonal demand planning.

---
*Notebook by Gouthami Mandapelli — AICTE/BharatSkills Internship Project*